In [1]:
import os, glob, gc, warnings
import numpy as np
import pandas as pd

PATH_DATASET = "/kaggle/input/icecube-neutrinos-in-deep-ice"
DATA_DIR = PATH_DATASET

warnings.filterwarnings("ignore")


def cartesian_to_sphere(x, y, z):
    r = np.sqrt(x * x + y * y + z * z)
    azimuth = np.arctan2(y, x)
    zenith = np.arccos(z / r)
    return azimuth, zenith


def adjust_sphere(azimuth, zenith):
    if zenith < 0:
        zenith += np.pi
        azimuth += np.pi
    if azimuth < 0:
        azimuth += 2 * np.pi
    azimuth = azimuth % (2 * np.pi)
    return azimuth, zenith




In [2]:
# load sensor geometry once
geometry = pd.read_csv(os.path.join(DATA_DIR, "sensor_geometry.csv"))
geometry.set_index("sensor_id", inplace=True)
geometry = geometry.apply(np.float32)



In [3]:
# sample submission template
ssub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
ssub.set_index("event_id", inplace=True)
ssub = ssub.apply(np.float32)



In [4]:
batch_files = sorted(glob.glob(os.path.join(PATH_DATASET, "test", "*.parquet")))

for batch_file in batch_files:
    print(f"processing: {batch_file}")
    df = pd.read_parquet(
        batch_file,
        columns=["event_id", "time", "sensor_id", "charge", "auxiliary"],
    )
    # remove auxiliary pulses
    df = df[~df["auxiliary"]]

    # merge geometry
    df = df.merge(geometry, left_on="sensor_id", right_index=True, how="left")

    # cast once
    df[["x", "y", "z", "time", "charge"]] = df[
        ["x", "y", "z", "time", "charge"]
    ].astype(np.float32, copy=False)

    # keep at most 800 highest‑charge pulses per event
    df = df.sort_values(["event_id", "charge"], ascending=[True, False])
    df = df.groupby("event_id", sort=False).head(800)

    # helper columns for vectorised aggregates
    df["rt_x"] = df["x"] * df["time"]
    df["rt_y"] = df["y"] * df["time"]
    df["rt_z"] = df["z"] * df["time"]
    df["t2"] = df["time"] * df["time"]

    # aggregate per event
    agg = df.groupby("event_id").agg(
        cnt=("time", "size"),
        sum_r_x=("x", "sum"),
        sum_r_y=("y", "sum"),
        sum_r_z=("z", "sum"),
        sum_t=("time", "sum"),
        sum_t2=("t2", "sum"),
        sum_rt_x=("rt_x", "sum"),
        sum_rt_y=("rt_y", "sum"),
        sum_rt_z=("rt_z", "sum"),
    )

    # compute averages
    cnt = agg["cnt"].values[:, None]  # (E,1)
    avg_r = np.stack([agg["sum_r_x"], agg["sum_r_y"], agg["sum_r_z"]], axis=1) / cnt
    avg_t = (agg["sum_t"] / agg["cnt"]).values[:, None]  # (E,1)
    avg_rt = np.stack([agg["sum_rt_x"], agg["sum_rt_y"], agg["sum_rt_z"]], axis=1) / cnt
    q = (agg["sum_t2"] / agg["cnt"]).values - (avg_t.squeeze()) ** 2  # (E,)

    # direction estimate (vectorised)
    v_est = (avg_rt - avg_r * avg_t) / q[:, None]  # (E,3)
    v_est = -v_est  # flip direction as original script

    # convert to spherical coordinates
    azimuth, zenith = cartesian_to_sphere(v_est[:, 0], v_est[:, 1], v_est[:, 2])
    # adjust angles
    adj = np.vectorize(adjust_sphere, otypes=[np.float32, np.float32])
    azimuth, zenith = adj(azimuth.astype(np.float32), zenith.astype(np.float32))

    # write back to submission DataFrame in bulk
    event_ids = agg.index
    ssub.loc[event_ids, "azimuth"] = azimuth
    ssub.loc[event_ids, "zenith"] = zenith

    # optional progress print for small batches
    if len(df) < 1e5:
        for eid, a, z in zip(event_ids[:5], azimuth[:5], zenith[:5]):
            print(f"Estimated event {eid}: azimuth={a:.4f}, zenith={z:.4f}")

    del df, agg
    gc.collect()



processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_104.parquet
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_128.parquet
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_141.parquet
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_143.parquet
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_15.parquet
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_160.parquet
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_163.parquet
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_171.parquet
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_176.parquet
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_201.parquet
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_207.parquet
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_211.parquet
processing: /kaggle/input/icecube-neutrinos-in-deep-i

In [5]:
ssub.fillna(0).to_csv("submission.csv", index=True)
print("Submission file written: submission.csv")

Submission file written: submission.csv
